In [5]:
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from unified_dataset import UnifiedIIDDataset, create_iid_data_loaders, custom_collate_fn

In [6]:
def visualize_batch(batch, save_path=None):
    """Visualize a batch of images"""
    batch_size = batch['rgb'].shape[0]
    
    # Create subplots
    fig, axes = plt.subplots(batch_size, 4, figsize=(16, 4 * batch_size))
    if batch_size == 1:
        axes = axes.unsqueeze(0)
    
    for i in range(batch_size):
        # RGB image
        rgb = batch['rgb'][i].permute(1, 2, 0).cpu().numpy()
        rgb = np.clip(rgb, 0, 1)
        axes[i, 0].imshow(rgb)
        axes[i, 0].set_title(f"RGB - {batch['dataset'][i]}")
        axes[i, 0].axis('off')
        
        # Albedo/Reflectance
        if 'albedo' in batch:
            albedo = batch['albedo'][i]
            if albedo.shape[0] == 1:  # Single channel
                albedo = albedo.repeat(3, 1, 1)
            albedo = albedo.permute(1, 2, 0).cpu().numpy()
            albedo = np.clip(albedo, 0, 1)
            axes[i, 1].imshow(albedo)
            axes[i, 1].set_title("Albedo/Reflectance")
            axes[i, 1].axis('off')
        else:
            axes[i, 1].text(0.5, 0.5, "No Albedo", ha='center', va='center')
            axes[i, 1].set_title("No Albedo")
            axes[i, 1].axis('off')
        
        # Shading
        if 'shading' in batch:
            shading = batch['shading'][i]
            if shading.shape[0] == 1:  # Single channel
                shading = shading.repeat(3, 1, 1)
            shading = shading.permute(1, 2, 0).cpu().numpy()
            shading = np.clip(shading, 0, 1)
            axes[i, 2].imshow(shading, cmap='gray')
            axes[i, 2].set_title("Shading")
            axes[i, 2].axis('off')
        else:
            axes[i, 2].text(0.5, 0.5, "No Shading", ha='center', va='center')
            axes[i, 2].set_title("No Shading")
            axes[i, 2].axis('off')
        
        # Scene info
        axes[i, 3].text(0.1, 0.8, f"Dataset: {batch['dataset'][i]}", fontsize=12)
        axes[i, 3].text(0.1, 0.6, f"Scene: {batch['scene'][i]}", fontsize=12)
        axes[i, 3].text(0.1, 0.4, f"File: {batch['filename'][i]}", fontsize=10)
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Visualization saved to {save_path}")
    else:
        plt.show()
    
    plt.close()

In [7]:
def test_dataset_loading():
    """Test dataset loading for each dataset individually"""
    
    # Test configuration
    test_config = {
        'datasets': {
            'midintrinsics': {
                'path': 'datasets/MIDIntrinsics',
                'enabled': True
            },
            'mit_intrinsic': {
                'path': 'datasets/MIT-intrinsic',
                'enabled': True
            },
            'mpi_sintel': {
                'path': 'datasets/MPI_Sintel',
                'enabled': True
            },
            'ned': {
                'path': 'datasets/NED',
                'enabled': True
            }
        },
        'image_size': 256,
        'batch_size': 6,
        'num_workers': 0,  # Use 0 for debugging
        'max_samples_per_dataset': 5  # Limit for testing
    }
    
    print("Testing dataset loading...")
    
    # Test each dataset individually
    for dataset_name in ['midintrinsics', 'mit_intrinsic', 'mpi_sintel', 'ned']:
        print(f"\n{'='*50}")
        print(f"Testing {dataset_name.upper()} dataset")
        print(f"{'='*50}")
        
        # Create config for single dataset
        single_dataset_config = test_config.copy()
        for key in single_dataset_config['datasets']:
            single_dataset_config['datasets'][key]['enabled'] = (key == dataset_name)
        
        try:
            # Create dataset
            dataset = UnifiedIIDDataset(
                datasets_config=single_dataset_config['datasets'],
                split='train',
                image_size=test_config['image_size'],
                max_samples_per_dataset=test_config['max_samples_per_dataset']
            )
            
            print(f"Dataset loaded successfully!")
            print(f"Number of samples: {len(dataset)}")
            
            if len(dataset) > 0:
                # Test loading a sample
                sample = dataset[0]
                print(f"Sample keys: {list(sample.keys())}")
                print(f"RGB shape: {sample['rgb'].shape}")
                if 'albedo' in sample:
                    print(f"Albedo shape: {sample['albedo'].shape}")
                if 'shading' in sample:
                    print(f"Shading shape: {sample['shading'].shape}")
                
                # Create data loader with custom collate function
                from torch.utils.data import DataLoader
                loader = DataLoader(
                    dataset,
                    batch_size=test_config['batch_size'],
                    shuffle=True,
                    num_workers=0,
                    collate_fn=custom_collate_fn
                )
                
                # Test batch loading
                for batch in loader:
                    print(f"Batch loaded successfully!")
                    print(f"Batch keys: {list(batch.keys())}")
                    print(f"Batch RGB shape: {batch['rgb'].shape}")
                    
                    # Visualize batch
                    output_dir = Path('test_outputs')
                    output_dir.mkdir(exist_ok=True)
                    save_path = output_dir / f"{dataset_name}_batch.png"
                    visualize_batch(batch, save_path)
                    break
            else:
                print("No samples found in dataset")
                
        except Exception as e:
            print(f"Error loading {dataset_name} dataset: {e}")
            import traceback
            traceback.print_exc()


In [8]:
test_dataset_loading()

Testing dataset loading...

Testing MIDINTRINSICS dataset
Loaded 4925 total samples for train split
Dataset distribution: {'midintrinsics': 4925}
Dataset loaded successfully!
Number of samples: 4925
Sample keys: ['rgb', 'dataset', 'scene', 'filename', 'albedo', 'shading', 'metadata']
RGB shape: torch.Size([3, 256, 256])
Albedo shape: torch.Size([3, 256, 256])
Shading shape: torch.Size([1, 256, 256])
Batch loaded successfully!
Batch keys: ['shading', 'albedo', 'filename', 'metadata', 'dataset', 'scene', 'rgb']
Batch RGB shape: torch.Size([6, 3, 256, 256])
Visualization saved to test_outputs/midintrinsics_batch.png

Testing MIT_INTRINSIC dataset
Loaded 100 total samples for train split
Dataset distribution: {'mit_intrinsic': 100}
Dataset loaded successfully!
Number of samples: 100
Sample keys: ['rgb', 'dataset', 'scene', 'filename', 'albedo', 'shading', 'metadata']
RGB shape: torch.Size([3, 256, 256])
Albedo shape: torch.Size([3, 256, 256])
Shading shape: torch.Size([1, 256, 256])
Batch 

In [ ]:
output_dir = Path('test_outputs')
output_dir.mkdir(exist_ok=True)

In [ ]:
test_dataset_loading()